# Smart Buildings — Forecast Ablation A/B/C/D

5 shipped scenarios x 8640 steps x 4 arms. Kaggle: Internet ON required.
Runtime ~1-2 h. Results land in `/kaggle/working/ablation_results/`.

In [ ]:
import pathlib, subprocess, os

URL  = "https://github.com/KAC-PDW/smart-buildings-competition.git"
REPO = pathlib.Path("/kaggle/working/smart-buildings-competition")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -e . --no-deps
!pip install -q timesfm

try:
    import numpy, numba
    print("numpy", numpy.__version__, "| numba ok")
except Exception as exc:
    print("numba broken, pinning numpy 2.0.2:", exc)
    !pip install -q "numpy==2.0.2"

import torch
print("cuda available:", torch.cuda.is_available())

In [ ]:
# 288-step sanity check, arm A only (fast)
!python smart_buildings_toolkit/scripts/run_experiment.py --mode smoke --arms A --skip-install

In [ ]:
# Full run. Add `--device cuda` if the notebook has a GPU accelerator.
!python smart_buildings_toolkit/scripts/run_experiment.py --mode rep5 --arms A B C D --extra-test-count 0 --skip-install

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/ablation_results/ablation_summary.csv")
piv = df.pivot_table(index="weather", columns="arm", values="mae_vs_zero")
print(piv.round(5).to_string())
print()
print(df.groupby("arm")["mae_vs_zero"].mean().round(5).to_string())